[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter1/sample-rag.ipynb)

# Chapter 1: Simple RAG Example with LangChain

This notebook demonstrates the basic flow of Retrieval-Augmented Generation (RAG) using LangChain:
1. **Load**: Download and parse a PDF document
2. **Split**: Chunk the document into smaller pieces
3. **Embed & Store**: Create embeddings and store in a vector database
4. **Query**: Retrieve relevant chunks and generate responses

**What you'll learn:**
- Build a basic RAG pipeline from scratch using LangChain
- Load, split, embed, and store PDF documents in a vector database (LanceDB)
- Create a retrieval-augmented generation chain using LCEL
- Query the system and get context-grounded answers

**Prerequisites:** `pip install -r requirements.txt` and set `OPENAI_API_KEY` in your environment.

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import LanceDB
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## Step 1: Load the PDF Document

We'll use LangChain's PyPDFLoader to download and parse the "Alice in Wonderland" PDF.

In [2]:
# Load the PDF from URL
pdf_url = "https://www.adobe.com/be_en/active-use/pdf/Alice_in_Wonderland.pdf"
loader = PyPDFLoader(pdf_url)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF")

Loaded 105 pages from the PDF


## Step 2: Split into Chunks

Long documents need to be split into smaller chunks for effective retrieval. We use `RecursiveCharacterTextSplitter` which tries to split on natural boundaries (paragraphs, sentences) while respecting the chunk size.

Here we use chunk size of 1000 characters with a 200 characters overlap.

In [3]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

chunks = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

Split into 245 chunks


> **Note:** A chunk size of 1,000 characters with 200-character overlap is a common starting point for text splitting. The overlap ensures that sentences spanning a chunk boundary aren't lost. Smaller chunks improve retrieval precision but may lack context; larger chunks retain more context but may dilute relevance signals. Tune these values based on your document structure and query patterns.

## Step 3: Create Embeddings and Vector Store

We embed all chunks using OpenAI's embedding model and store them in LanceDB, a serverless vector database with persistent storage.

In [4]:
# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = LanceDB.from_documents(chunks, embeddings)

print(f"Created vector store with {len(chunks)} documents")

Created vector store with 245 documents


## Step 4: RAG Query - Retrieval + Generation

Using LCEL (LangChain Expression Language), we create a chain that:
1. Takes a question
2. Retrieves relevant documents
3. Formats them as context
4. Passes to the LLM for answering

In [5]:
# Create the RAG chain using LCEL (LangChain Expression Language)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}"""
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [6]:
# Ask questions about Alice in Wonderland
questions = [
    "Who is the main character and what happens at the beginning of the story?",
    "What did the Caterpillar ask Alice?",
    "Describe the Mad Hatter's tea party.",
    "What happened at the trial at the end of the story?"
]

for q in questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

Q: Who is the main character and what happens at the beginning of the story?
A: The main character is Alice. At the beginning of the story, the White Rabbit puts on his spectacles and asks the King where he should begin. The King instructs him to "begin at the beginning" and to continue until he reaches the end, then to stop.
------------------------------------------------------------
Q: What did the Caterpillar ask Alice?
A: The Caterpillar asked Alice, "Who are you?"
------------------------------------------------------------
Q: Describe the Mad Hatter's tea party.
A: The Mad Hatter's tea party is a chaotic and nonsensical gathering set under a tree, where the March Hare, the Hatter, and a Dormouse are present. The table is large but crowded, with the three characters squeezed together at one corner. The Dormouse is asleep, and the other two use it as a cushion while they converse. The atmosphere is filled with rudeness and absurdity, as Alice, who joins the party, finds herself in